# HTTM ThreeBehavior V2 ? Independent Test Evaluation (no training)

This notebook only evaluates `httm_three_behavior_v2_best.pt` on a separate Test package. It never trains and never substitutes Validation for Test. Annotation completeness must be explicitly verified; missing/empty labels are not treated as negatives. Current source audit status: **TEST_NOT_AVAILABLE**.

Attach the V2 training artifacts ZIP (or raw checkpoint) and, when available, a separately prepared Test ZIP containing `readiness.json`, `data.yaml`, and `test_manifest.csv`. The Test readiness must set `test_evaluation_ready: true`, `annotation_completeness: VERIFIED`; each row must have `review_status: APPROVED_TEST_GROUND_TRUTH`, `annotation_completeness: VERIFIED`, `split: test`, scene group, image SHA-256 and pHash. Attach the V2 dataset package too; its `source_manifest.csv` supplies Train/Validation hashes and scene groups for leakage checks.


In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, zipfile
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working/httm_v2_test_evaluation')
WORK.mkdir(parents=True, exist_ok=True)
CLASSES = {'0': 'sleeping', '1': 'using_phone', '2': 'raising_hand'}

# Locate the trained checkpoint directly or inside its Kaggle artifact ZIP.
checkpoint_paths = list(INPUT.rglob('httm_three_behavior_v2_best.pt')) if INPUT.exists() else []
checkpoint = checkpoint_paths[0] if checkpoint_paths else None
if checkpoint is None and INPUT.exists():
    for archive in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as zf:
                member = next((n for n in zf.namelist() if Path(n).name == 'httm_three_behavior_v2_best.pt'), None)
                if member:
                    checkpoint = WORK / 'httm_three_behavior_v2_best.pt'
                    checkpoint.write_bytes(zf.read(member))
                    break
        except zipfile.BadZipFile:
            continue
if checkpoint is None:
    raise FileNotFoundError('Attach the V2 training_artifacts.zip or httm_three_behavior_v2_best.pt to Kaggle Input.')
print('checkpoint=', checkpoint)

# Find the exact V2 Train/Validation manifest used for leakage exclusion.
v2_manifest = None
if INPUT.exists():
    for manifest in INPUT.rglob('source_manifest.csv'):
        try:
            with manifest.open(encoding='utf-8-sig', newline='') as f:
                sample = next(csv.DictReader(f), None)
            # Accept only the exact V2 Train/Validation package, never another manifest.
            ready_path = manifest.parent / 'readiness.json'
            if sample and ready_path.is_file() and {'split','scene_group','image_sha256','image_phash'} <= set(sample):
                meta = json.loads(ready_path.read_text(encoding='utf-8'))
                if (meta.get('classes') == CLASSES and meta.get('image_counts',{}).get('train') == 1086
                    and meta.get('image_counts',{}).get('val') == 214):
                    v2_manifest = manifest
                    break
        except (OSError, ValueError, StopIteration):
            continue

# A Test package is accepted only when it carries explicit verified-GT readiness.
TEST_ROOT = None
TEST_READY = False
BLOCKERS = []
if not v2_manifest:
    BLOCKERS.append('V2 Train/Validation source_manifest.csv is not attached; cannot check split leakage.')
if INPUT.exists():
    for readiness_path in INPUT.rglob('readiness.json'):
        try:
            obj = json.loads(readiness_path.read_text(encoding='utf-8'))
        except (OSError, ValueError):
            continue
        if (obj.get('test_evaluation_ready') is True
            and obj.get('annotation_completeness') == 'VERIFIED'
            and obj.get('classes') == CLASSES
            and (readiness_path.parent / 'data.yaml').is_file()
            and (readiness_path.parent / 'test_manifest.csv').is_file()):
            TEST_ROOT = readiness_path.parent
            break
if TEST_ROOT is None and INPUT.exists():
    # Kaggle datasets do not necessarily expand uploaded ZIP files automatically.
    for archive in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive) as zf:
                readiness_member = None
                for member in zf.namelist():
                    if Path(member).name != 'readiness.json': continue
                    try: candidate=json.loads(zf.read(member).decode('utf-8'))
                    except (ValueError, UnicodeDecodeError): continue
                    prefix=member.rsplit('/',1)[0] if '/' in member else ''
                    siblings={n[len(prefix)+1:] if prefix else n for n in zf.namelist()}
                    if (candidate.get('test_evaluation_ready') is True
                        and candidate.get('annotation_completeness') == 'VERIFIED'
                        and candidate.get('classes') == CLASSES
                        and 'data.yaml' in siblings and 'test_manifest.csv' in siblings):
                        readiness_member=member; break
                if readiness_member is None: continue
                extract_root=WORK/'test_package'; extract_root.mkdir(parents=True,exist_ok=True)
                for info in zf.infolist():
                    if info.is_dir(): continue
                    target=(extract_root/info.filename).resolve()
                    if extract_root.resolve() not in target.parents:
                        raise RuntimeError(f'Unsafe ZIP member path: {info.filename}')
                    target.parent.mkdir(parents=True,exist_ok=True)
                    target.write_bytes(zf.read(info.filename))
                prefix=readiness_member.rsplit('/',1)[0] if '/' in readiness_member else ''
                TEST_ROOT=(extract_root/prefix) if prefix else extract_root
                break
        except zipfile.BadZipFile:
            continue
if TEST_ROOT is None:
    BLOCKERS.append('No separate three-class Test package declares verified annotation completeness and approved Test ground truth.')

if not BLOCKERS:
    with (v2_manifest).open(encoding='utf-8-sig', newline='') as f:
        trainval_rows = [r for r in csv.DictReader(f) if r.get('split') in ('train','val')]
    with (TEST_ROOT/'test_manifest.csv').open(encoding='utf-8-sig', newline='') as f:
        test_rows = list(csv.DictReader(f))
    required = {'split','scene_group','image_sha256','image_phash','label_sha256','annotation_completeness','review_status','image_rel','label_rel'}
    if not test_rows or not required <= set(test_rows[0]):
        BLOCKERS.append('Test manifest is empty or lacks required provenance/completeness/hash columns.')
    else:
        train_groups = {r['scene_group'] for r in trainval_rows}
        test_groups = {r['scene_group'] for r in test_rows}
        train_sha = {r['image_sha256'] for r in trainval_rows}
        train_ph = [int(r['image_phash'],16) for r in trainval_rows]
        if not test_groups or '' in test_groups or train_groups & test_groups:
            BLOCKERS.append('Scene groups are missing or overlap V2 Train/Validation.')
        if test_rows and any(r['split'] != 'test' or r['annotation_completeness'] != 'VERIFIED' or r['review_status'] != 'APPROVED_TEST_GROUND_TRUTH' for r in test_rows):
            BLOCKERS.append('Every row must be an approved, completeness-verified Test image.')
        if test_rows and (any(not r['image_sha256'] or not r['image_phash'] for r in test_rows)
                          or any(r['image_sha256'] in train_sha for r in test_rows)):
            BLOCKERS.append('Missing image hashes or exact-duplicate leakage against Train/Validation.')
        if test_rows and not BLOCKERS:
            near_pairs=[]
            for row in test_rows:
                hp=int(row['image_phash'],16)
                if any((hp ^ old).bit_count() <= 4 for old in train_ph): near_pairs.append(row['image_rel'])
            if near_pairs: BLOCKERS.append(f'pHash<=4 leakage against Train/Validation: {near_pairs[:5]}')
        TEST_READY = not BLOCKERS

print('test_status=', 'TEST_READY' if TEST_READY else 'TEST_NOT_AVAILABLE')
if BLOCKERS: print('blockers=', json.dumps(BLOCKERS, indent=2))


In [ ]:
# Evaluate on Test only. This cell has no training call and does not fall back to Validation.
if not TEST_READY:
    status = {'status':'TEST_NOT_AVAILABLE','metrics':None,'test_evaluated':False,'blockers':BLOCKERS}
    (WORK/'test_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
    print('TEST_NOT_AVAILABLE: evaluation skipped; Validation will not be substituted.')
else:
    try:
        import ultralytics
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','install','--quiet','ultralytics'])
    import math, yaml
    from ultralytics import YOLO
    data_cfg = yaml.safe_load((TEST_ROOT/'data.yaml').read_text(encoding='utf-8'))
    names = data_cfg.get('names',{})
    names = {str(k):str(v) for k,v in (names.items() if isinstance(names,dict) else enumerate(names))}
    if data_cfg.get('nc') != 3 or names != CLASSES:
        raise RuntimeError(f'Unexpected Test class mapping: nc={data_cfg.get("nc")}, names={names}')
    # Validate all Test images/labels, and require positive instances for all target classes.
    images_root = TEST_ROOT / data_cfg['test']
    labels_root = TEST_ROOT / 'labels' / 'test'
    test_images = sorted(p for p in images_root.rglob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'})
    if not test_images or len(test_images) != len(test_rows):
        raise RuntimeError('Test image count does not match the approved manifest.')
    class_ids=set(); EPS=1e-6
    for image in test_images:
        label=labels_root/(image.stem+'.txt')
        if not label.is_file() or not label.read_text(encoding='utf-8').strip():
            raise RuntimeError(f'Missing/empty Test label: {label}')
        for line_no,line in enumerate(label.read_text(encoding='utf-8').splitlines(),1):
            parts=line.split()
            if len(parts)!=5: raise RuntimeError(f'{label}:{line_no}: invalid YOLO row')
            c=int(parts[0]); x,y,w,h=map(float,parts[1:]); bounds=(x-w/2,y-h/2,x+w/2,y+h/2)
            if c not in (0,1,2) or not all(map(math.isfinite,(x,y,w,h))) or w<=0 or h<=0 or min(bounds[:2]) < -EPS or max(bounds[2:]) > 1+EPS:
                raise RuntimeError(f'{label}:{line_no}: invalid class/bbox {parts}')
            class_ids.add(c)
    if class_ids != {0,1,2}: raise RuntimeError(f'Test must contain all three labeled classes; found {class_ids}')
    rows_by_image={Path(r['image_rel']).name:r for r in test_rows}
    for image in test_images:
        row=rows_by_image.get(image.name)
        if row is None: raise RuntimeError(f'Image absent from approved manifest: {image.name}')
        label=labels_root/(image.stem+'.txt')
        if hashlib.sha256(image.read_bytes()).hexdigest()!=row['image_sha256']:
            raise RuntimeError(f'Test image hash mismatch: {image}')
        if hashlib.sha256(label.read_bytes()).hexdigest()!=row['label_sha256']:
            raise RuntimeError(f'Test label hash mismatch: {label}')
    runtime_cfg=dict(data_cfg)
    runtime_cfg['path']=str(TEST_ROOT.resolve())
    v2_root=v2_manifest.parent
    runtime_cfg['train']=str((v2_root/'images/train').resolve())
    runtime_cfg['val']=str((v2_root/'images/val').resolve())
    runtime_cfg['test']=str(images_root.resolve())
    runtime_yaml=WORK/'test_data_runtime.yaml'
    runtime_yaml.write_text(yaml.safe_dump(runtime_cfg,sort_keys=False),encoding='utf-8')
    model=YOLO(str(checkpoint))
    model_names=model.names
    model_names={str(k):str(v) for k,v in (model_names.items() if isinstance(model_names,dict) else enumerate(model_names))}
    if model_names != CLASSES: raise RuntimeError(f'Checkpoint class mapping mismatch: {model_names}')
    metrics=model.val(data=str(runtime_yaml),split='test',imgsz=640,plots=True,verbose=False)
    box=metrics.box
    per_class={}
    for i,class_id in enumerate(list(box.ap_class_index)):
        per_class[CLASSES[str(int(class_id))]]={'precision':float(box.p[i]),'recall':float(box.r[i]),'mAP50':float(box.ap50[i]),'mAP50_95':float(box.ap[i].mean())}
    pred_root=WORK/'test_prediction_samples'
    model.predict(source=[str(p) for p in test_images[:24]],imgsz=640,save=True,project=str(pred_root),name='samples',exist_ok=True,verbose=False)
    output=WORK/'test_evaluation_artifacts'; output.mkdir(exist_ok=True)
    for plot in Path(metrics.save_dir).glob('*.png'): shutil.copy2(plot,output/plot.name)
    shutil.copytree(pred_root/'samples',output/'prediction_samples',dirs_exist_ok=True)
    result={'status':'TEST_EVALUATED','test_evaluated':True,'annotation_completeness':'VERIFIED','split':'test','per_class_metrics':per_class,'test_images':len(test_images),'checkpoint':str(checkpoint),'warning':'Metrics describe only the approved supplied Test package.'}
    (output/'metrics.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    artifacts=WORK/'HTTM_V2_test_evaluation_artifacts.zip'
    with zipfile.ZipFile(artifacts,'w',zipfile.ZIP_DEFLATED) as zf:
        for p in output.rglob('*'):
            if p.is_file(): zf.write(p,p.relative_to(output).as_posix())
    print(json.dumps(result,indent=2)); print('artifacts=',artifacts)
